# Iterative Private Gaussian Graphical Model Implementation

This notebook implements the iterative version of Private-GGM (Algorithm 1) from the paper. The algorithm:
1. Splits privacy budget between initialization and iterative measurements
2. Uses exponential mechanism to select next measurement
3. Applies Gaussian mechanism for private measurements
4. Updates graph structure iteratively

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import optax
from privateggm.model import MaxEntropy
from privateggm.data import generate_data
from privateggm.utils import get_K, get_z, initialize_z

# Set random seed for reproducibility
np.random.seed(42)
key = jax.random.PRNGKey(42)

ModuleNotFoundError: No module named 'privateggm'

## 1. Initialization and Setup

First, we'll implement the budget splitting and initialization phase. From Algorithm 1:
- ρ_m = ρα/2 : mean estimation
- ρ_v = ρα/2 : variance estimation (diagonal)
- ρ_c = ρ(1-α)/2 : covariance estimation
- ρ_s = ρ(1-α)/2 : selection mechanism

In [ ]:
class IterativePrivateGGM:
    def __init__(self, n_features, epsilon, delta, alpha=0.2, T=10):
        """
        Initialize Iterative Private GGM
        
        Args:
            n_features (int): Number of features (dimension)
            epsilon (float): Privacy parameter
            delta (float): Privacy parameter
            alpha (float): Budget split parameter (default 0.2)
            T (int): Number of iterations
        """
        self.d = n_features
        self.epsilon = epsilon
        self.delta = delta
        self.alpha = alpha
        self.T = T
        
        # Split privacy budget
        self.rho = epsilon**2 / (2 * np.log(1/delta))  # Convert to zCDP
        self.rho_m = self.rho * alpha / 2  # mean estimation
        self.rho_v = self.rho * alpha / 2  # variance estimation
        self.rho_c = self.rho * (1-alpha) / 2  # covariance estimation
        self.rho_s = self.rho * (1-alpha) / 2  # selection mechanism
        
        # Initialize storage for measurements
        self.measured_entries = set()  # Track which entries we've measured
        self.S_tilde = np.zeros((n_features, n_features))  # Noisy measurements

## 2. Privacy Mechanisms

Implement the Gaussian mechanism for zCDP and the exponential mechanism for entry selection.

In [ ]:
def gaussian_mechanism(self, value, sensitivity, rho):
    """
    Implement Gaussian mechanism for zCDP
    
    Args:
        value: True value to privatize
        sensitivity: L2 sensitivity of the query
        rho: Privacy parameter for this measurement
    """
    sigma = sensitivity / np.sqrt(2 * rho)
    noise = np.random.normal(0, sigma)
    return value + noise

def exponential_mechanism(self, scores, sensitivity, rho):
    """
    Implement exponential mechanism for selecting next measurement
    
    Args:
        scores: Array of scores for each possible measurement
        sensitivity: Sensitivity of the score function
        rho: Privacy parameter for this selection
    """
    epsilon = np.sqrt(2 * rho)  # Convert from zCDP to approximate DP
    scaled_scores = epsilon * scores / (2 * sensitivity)
    probabilities = np.exp(scaled_scores)
    probabilities = probabilities / np.sum(probabilities)
    
    # Sample an index according to the probabilities
    return np.random.choice(len(scores), p=probabilities)

## 3. Score Function for Entry Selection

Implement the score function used in the exponential mechanism to select the next entry to measure.

In [ ]:
def compute_entry_scores(self, X, Sigma_hat):
    """
    Compute scores for each unmeasured entry to determine next measurement
    
    Args:
        X: Data matrix
        Sigma_hat: Current estimate of covariance matrix
    """
    n = X.shape[0]
    empirical_cov = (X.T @ X) / n
    scores = np.zeros((self.d, self.d))
    
    # Score is absolute difference between empirical and current estimate
    for i in range(self.d):
        for j in range(i+1):
            if (i,j) not in self.measured_entries:
                scores[i,j] = np.abs(empirical_cov[i,j] - Sigma_hat[i,j])
                scores[j,i] = scores[i,j]  # Symmetric
                
    return scores

## 4. Main Iterative Algorithm

Implement the main iterative algorithm that:
1. Initializes with private mean and diagonal entries
2. Iteratively selects and measures entries
3. Updates the graph structure

In [ ]:
def fit(self, X):
    """
    Fit the private GGM model to data
    
    Args:
        X: Data matrix of shape (n_samples, n_features)
    """
    n, d = X.shape
    assert d == self.d
    
    # 1. Initialize with private mean (not using it for now, assuming centered)
    
    # 2. Private diagonal entries
    diag_sensitivity = np.sqrt(2) / n  # For zero-mean Gaussian bounded by 1
    for i in range(d):
        empirical_var = np.sum(X[:,i]**2) / n
        self.S_tilde[i,i] = self.gaussian_mechanism(
            empirical_var, 
            diag_sensitivity, 
            self.rho_v / d
        )
        self.measured_entries.add((i,i))
    
    # Initialize MaxEnt solver
    self.maxent = MaxEntropy()
    
    # 3. Iterative measurements
    rho_per_iter = self.rho_c / self.T  # Budget per iteration
    
    for t in range(self.T):
        # Get current estimate
        Sigma_hat = self.maxent.estimate(self.S_tilde)
        
        # Compute scores and select entry
        scores = self.compute_entry_scores(X, Sigma_hat)
        score_sensitivity = 2 / n  # Need to verify this
        flat_scores = scores.reshape(-1)
        selected_idx = self.exponential_mechanism(
            flat_scores, 
            score_sensitivity, 
            self.rho_s / self.T
        )
        i, j = np.unravel_index(selected_idx, (d,d))
        
        # Measure selected entry
        cov_sensitivity = 1 / n  # For zero-mean Gaussian bounded by 1
        empirical_cov = np.sum(X[:,i] * X[:,j]) / n
        self.S_tilde[i,j] = self.gaussian_mechanism(
            empirical_cov,
            cov_sensitivity,
            rho_per_iter
        )
        self.S_tilde[j,i] = self.S_tilde[i,j]  # Symmetric
        self.measured_entries.add((i,j))
        self.measured_entries.add((j,i))
    
    # Final estimate
    self.Sigma_hat = self.maxent.estimate(self.S_tilde)
    return self.Sigma_hat

## 5. Testing the Implementation

Let's test the implementation on synthetic data:
1. Generate synthetic data with known structure
2. Apply our iterative private GGM
3. Compare results with non-private and true covariance

In [ ]:
# Generate synthetic data
n_samples = 1000
n_features = 5
sparsity = 0.3

# Generate synthetic sparse precision matrix and data
X, true_cov, true_prec = generate_data(
    n_samples=n_samples,
    d=n_features,
    off_diag_perc=sparsity,
    rescale=True
)

# Initialize and fit our model
model = IterativePrivateGGM(
    n_features=n_features,
    epsilon=1.0,
    delta=1e-6,
    alpha=0.2,
    T=10
)

# Fit the model
Sigma_hat = model.fit(X)

# Compare results
print("True Covariance:")
print(true_cov)
print("\nEstimated Covariance:")
print(Sigma_hat)

## 6. Analysis Functions

Add helper functions to analyze the results:
1. Measure entry-wise error
2. Track privacy budget usage
3. Visualize convergence

In [ ]:
def analyze_results(true_cov, est_cov, measured_entries):
    """Analyze the quality of the estimation"""
    # Frobenius norm error
    frob_error = np.linalg.norm(true_cov - est_cov, 'fro')
    
    # Spectral norm error
    spec_error = np.linalg.norm(true_cov - est_cov, 2)
    
    # Element-wise maximum error
    max_error = np.max(np.abs(true_cov - est_cov))
    
    # Coverage of measurements
    coverage = len(measured_entries) / (true_cov.shape[0]**2)
    
    return {
        'frobenius_error': frob_error,
        'spectral_error': spec_error,
        'max_error': max_error,
        'coverage': coverage
    }

# Analyze results
results = analyze_results(true_cov, Sigma_hat, model.measured_entries)
for metric, value in results.items():
    print(f"{metric}: {value}")